- A real project would use hundreds or thousands of labeled reviews, such as the IMDb review dataset.

In [ ]:
# Hugging Face datasets library is not currently installed in your Python environment. Because it is a third-party library 
# and not built into standard Python, you have to download it first using below command:
# - if not installed
# !pip install datasets

In [ ]:
from datasets import load_dataset
# older tutorial, the single-word "imdb" used to work perfectly.but now it requires the namespace "stanfordnlp/" to be added to the dataset name.
ds = load_dataset("stanfordnlp/imdb") # Added the 'stanfordnlp/' namespace to the dataset name

X_train_text = ds["train"]["text"]
y_train = ds["train"]["label"]
X_test_text = ds["test"]["text"]
y_test = ds["test"]["label"]

- TextVectorization expects raw text strings as input, but imdb.load_data() already gives you integers. 
- You have two options: decode back to text (for learning purposes) or switch to the raw-text version of IMDB (the practical way). 
- here show the one with raw texts:

Why we can't use it(TextVectorization) as-is

- TextVectorization is a preprocessing layer, not a decoder.
- Its job is: raw string → tokens → integers. 
- My X_train is already at the "integers" end of that pipeline, so there's nothing left for it to vectorize.

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization, Embedding, SimpleRNN, Dense
from tensorflow.keras.models import Sequential
VOCAB_SIZE =10000
maxlen = 200

vectorize_layer = TextVectorization(
    max_tokens = VOCAB_SIZE,
    output_mode = 'int',
    output_sequence_length = maxlen # replace pad sequence entirely.
)

# "adapt" builds the vocabulary by scanning the training text —
# this is the step that replaces Tokenizer().fit_on_texts()

vectorize_layer.adapt(X_train_text) 

# single adapt() call handles what previously took three separate pieces: lowercasing, 
# punctuation stripping, and vocabulary building 
# — TextVectorization does default standardization (lowercase + strip punctuation) automatically.

In [ ]:
# here we check the word indexing using the vectorize layer.
vocab = vectorize_layer.get_vocabulary()
print(f"Vocabulary size: {len(vocab)}")
print(f"First 10 words in the vocabulary: {vocab[:10]}")

In [ ]:
# index --> word
print(f"Word for index 0: {vocab[90]}")

# word -> index
print(f"Index for word 'movie': {vocab.index('good')}")

# counts occurrences of the word 'movie' in the vocabulary and returns its index.
print(f"count for word 'movie': {vocab.count('good')}")

In [ ]:
# index 0 = padding, index 1 = "[UNK]" (out-of-vocab) 
# same convention as imdb.load_data()'s reserved indices.
# Rest of the vocab is ordered by frequency (most common word = index 2).
# To see how a sentence gets encoded: vectorize_layer(["this movie was great"]).

- big advantage: we can put TextVectorization inside the model itself, 
- so the model accepts raw strings directly. 
- no separate preprocessing pipeline to maintain at inference time.

In [ ]:
model = Sequential([
    vectorize_layer,  # raw text input accepted directly.
    Embedding(VOCAB_SIZE, output_dim=32),
    SimpleRNN(32, return_sequences=True),
    SimpleRNN(32),   # second RNN, return_sequences=False (default) — collapses it.
    Dense(1, activation='sigmoid')
])

### Embedding(input_dim=vocab_size, output_dim=32)
- output_dim=32 → how many numbers represent each word. 
- Every one of those 10,000 words gets mapped to a 32-dimensional vector of learned floats. 
- This is a design choice — could be 16, 50, 100, etc.
-  Bigger = more expressive but more parameters to learn.
### Shape-wise, in your pipeline
So a single review goes from 200 integers → a 200 × 32 matrix.
- That matrix is what actually gets fed into SimpleRNN(32).
- which then reads it word-vector by word-vector across the sequence.

### Total learnable parameters this layer adds

- vocab_size × output_dim = 10,000 × 32 = 320,000 parameters — all learned from scratch during training on your 20,000 training reviews (unless you swap in pretrained vectors like GloVe later).

### Why not just feed the raw integers into the RNN?

- Because integer IDs imply a false relationship. If "great" = 530 and "terrible" = 531, 
- The model would wrongly think they're "close" just because their IDs are adjacent — but that's just where they landed in the frequency ranking, not their meaning.
-  The embedding layer replaces that arbitrary ID with a learned vector, and during training, words with similar sentiment/usage naturally end up with similar vectors.

In [ ]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
# This single line tells TensorFlow/Keras:
#  exactly which mathematical tools to use during the training phase.
# Adaptive Moment Estimation (Adam) optimizer is a popular choice for training deep learning models.

# Telling the model to track accuracy means that during training, 
# Keras will print out the percentage of images or texts
#  it correctly classified in that specific batch.

### Below input terms used in the ``model.fit()`` explained one by one.

``tf.constant()``:

- takes your standard Python list or array and converts it into a Tensor (TensorFlow's native, highly optimized data structure). 
- While Keras can often convert lists automatically, 
- doing it explicitly guarantees the ``data is formatted perfectly for GPU processing``.
- ``tf.constant(y_train)`` similarly for this.

In [ ]:
# The actual iterative process of feeding data, calculating errors, 
# and updating weights only happens later when we call:
# -  model.fit().

history = model.fit(
    tf.constant(X_train_text), tf.constant(y_train),
    epochs=5, batch_size=128, validation_split=0.2
)

## Quick Comparison

| Feature | Old Way (Tokenizer + pad_sequences) | TextVectorization |
|--------|--------------------------------------|--------------------|
| Where it lives | Separate preprocessing step | A layer inside the model |
| Input at inference | Must preprocess manually first | Accepts raw strings directly |
| Status | Legacy API | Current recommended approach |
| Saving | Must pickle the tokenizer separately | Saved automatically with the model |


## Step 1 — Evaluate on the test set

In [ ]:
loss, accuracy = model.evaluate(tf.constant(X_test_text), tf.constant(y_test))
print(f"Test loss: {loss:.4f}")
print(f"Test accuracy: {accuracy:.4f}")

### Step 2 — Generate predictions

### Explanation, How it ``(preds = (probs > 0.5).astype("int32").flatten())`` convert the preds in 1/0 labels:
- if the probability is greater than 50% (> 0.5), it evaluates to True.
- If it is 50% or less, it evaluates to False.
- Intermediate Result: An array of booleans like [[True], [False], [True]].

### 2. ``.astype("int32")`` : The Boolean Conversion
- Most evaluation metrics (like Scikit-Learn's accuracy_score or confusion matrices) cannot read True and False; they require standard numbers.
- This command forces the boolean array to change its data type to 32-bit integers.
- In Python and NumPy, True is mathematically converted to 1 and False is converted to 0.
- Intermediate Result: An array of integers like [[1], [0], [1]].
### 3. ``.flatten() : The Shape Correction``
- The .flatten() command takes the nested 2D array and collapses it into a flat, 1-dimensional array.
- Final Result: [1, 0, 1]

In [ ]:
probs = model.predict(tf.constant(X_test_text))   # shape (25000, 1), values 0–1
preds = (probs > 0.5).astype("int32").flatten()    # convert to 0/1 labels

### Step 3 — Inspect individual examples

In [ ]:
for i in range(5):
    print(f"Review:{X_test_text[i][:150]}....")
    print(f"True label:{y_test[i]} | predicted label:{preds[i]} |confidence : {probs[i][0]:.4f}")
    print("-"*80)    

### Step 4 — Test on My own sentences

In [ ]:
custom_reviews = [
    "This movie was absolutely fantastic, I loved every minute of it!",
    "Terrible film, complete waste of time and money.",
    "It was okay, not great but not bad either."
]

custom_probs = model.predict(tf.constant(custom_reviews))
for review, prob in zip(custom_reviews, custom_probs):
    label = "Positive" if prob > 0.5 else "Negative"
    print(f"'{review}' -> {label} ({prob[0]:.3f})")

### Step 5 — Reusable helper function

In [ ]:
def predict_sentiment(text):
    prob = model.predict(tf.constant([text]), verbose=0)[0][0]
    label = "Positive" if prob > 0.5 else "Negative"
    return label, float(prob)

label, confidence = predict_sentiment("The acting was superb but the plot dragged.")
print(f"{label} ({confidence:.3f})")

### Step 6 — Look at misclassified examples

In [ ]:
import numpy as np

y_test_arr = np.array(y_test)
mistakes = np.where(preds != y_test_arr)[0] # this returns a tuple of arrays like this: (array([1, 3]),) <- a TUPLE containing one array,
# we want the first array which contains the indices of the mistakes.

print(f"Total mistakes: {len(mistakes)} / {len(y_test_arr)}")

for i in mistakes[:5]:
    print(f"True: {y_test_arr[i]} | Predicted: {preds[i]} | Confidence: {probs[i][0]:.3f}")
    print(X_test_text[i][:200])
    print("-" * 80)

### Disadvantage of the ``Simple RNN:``
- **Vanishing gradients on long sequences** — during backpropagation through time, gradients shrink multiplicatively at each timestep, so by the time you're back at word 1 of a 200-word review, the gradient signal is nearly zero and those early words barely get learned from.
- **Can't retain long-range context** — practically, SimpleRNN "forgets" information from more than ~10-20 timesteps ago, which is why negation or setup ("the story starts great, but...") 150 words earlier often gets lost by the time the model reaches the end.
- **No gating mechanism** — unlike LSTM/GRU, it has no learned way to decide what to keep, update, or discard from its hidden state; every new word overwrites the state in the same fixed way regardless of relevance.
- **Sequential computation = slow training** — each timestep depends on the previous one's output, so it can't be parallelized across the sequence like a Transformer or CNN can, making it slower to train on long text.
- **Exploding gradients are also possible** — the same repeated-multiplication mechanic that causes vanishing gradients can instead blow up if weights are slightly too large, making training unstable without careful initialization or gradient clipping.
- **Lower accuracy ceiling on real tasks** — as you saw directly: ~75-83% on IMDB vs. ~87-90% for LSTM/GRU with almost no code changes, which is the practical reason SimpleRNN is rarely used in production, mainly serving as a teaching baseline.

### ``conclusion`` and ``motivation`` to apply advanced techniques:

- **Vanishing gradients limit SimpleRNN on long reviews** — by word 150–200, the hidden state has largely "forgotten" what happened at word 1, so context from earlier in a review gets lost before it can influence the prediction.
- **LSTM/GRU add gating mechanisms** (forget/input/output gates) that let the network selectively keep or discard information across many timesteps — directly fixing the problem you just saw in your misclassified examples.
- **Same code, bigger gain** — swapping `SimpleRNN(32)` for `LSTM(32)` or `GRU(32)` requires almost no changes to your existing pipeline, but typically lifts accuracy from ~75–83% to ~87–90% on IMDB.
- **Better at exactly what tripped up your model** — negation ("not bad at all") and longer mixed-sentiment reviews are the textbook cases where gated units outperform plain RNNs, since they can carry a "flip" signal (like "not") across several words without it decaying.